In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df=pd.read_csv('decision_tree_dataset.csv')
df.head()

,age,income,credit_score,city,gender,purchases,loan_amount,approved
0,41,16832.970365,646.435852,Hyderabad,Female,2,NaN,0
1,61,24958.920783,772.103370,Bangalore,Female,2,109037.369348,0
2,29,26661.565307,742.165011,Bangalore,Male,3,140981.063625,0
3,40,76918.367953,789.568381,Mumbai,Female,2,265765.286243,0
4,18,33446.160242,650.679984,Delhi,Male,4,187481.809910,0


In [4]:

#check the missing values
df.isnull().sum()

age               0
income          103
credit_score    104
city              0
gender            0
purchases         0
loan_amount     103
approved          0
dtype: int64

In [5]:
# finding outliers
df['city'].value_counts()


city
Delhi        268
Bangalore    262
Mumbai       261
Hyderabad    259
Name: count, dtype: int64

In [6]:
categorical_cols = df.select_dtypes(include=["object", "category"]).columns
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns

print("Categorical Columns:")
print(list(categorical_cols))

print("\nNumerical Columns:")
print(list(numerical_cols))

Categorical Columns:
['city', 'gender']

Numerical Columns:
['age', 'income', 'credit_score', 'purchases', 'loan_amount', 'approved']


In [7]:
df.isnull().sum()

age               0
income          103
credit_score    104
city              0
gender            0
purchases         0
loan_amount     103
approved          0
dtype: int64

In [10]:
median_income=df['income'].median()
print("median : " ,median_income)



mean :  55807.648029033655
median :  50967.1162221518


In [11]:
Q1=df['income'].quantile(0.25)
Q3=df['income'].quantile(0.75)
IQR=Q3-Q1
Lower = Q1 - 1.5*IQR
Upper = Q3 - 1.5*IQR
outliers=df[ (df['income']<Lower) | (df['income']>Upper) ]
outliers

,age,income,credit_score,city,gender,purchases,loan_amount,approved
3,40,76918.367953,789.568381,Mumbai,Female,2,265765.286243,0
4,18,33446.160242,650.679984,Delhi,Male,4,187481.809910,0
5,57,60504.648191,781.491445,Delhi,Female,3,223926.858722,1
6,29,39677.744822,612.480362,Bangalore,Female,4,178370.664259,0
7,62,58998.930950,734.910211,Hyderabad,Female,2,252981.038573,0
...,...,...,...,...,...,...,...,...
1045,57,86596.286095,548.490581,Bangalore,Male,3,124867.554418,0
1046,67,55567.188101,NaN,Delhi,Female,4,19776.528421,1
1047,43,57155.612408,773.613072,Mumbai,Female,1,195876.623311,0
1048,60,35425.789422,549.604263,Mumbai,Male,4,115295.988140,0


In [18]:
median_credit_score=df['credit_score'].median()
print("median:",median_credit_score)

median: 650.0527958856597


In [12]:
Q1=df['credit_score'].quantile(0.25)
Q3=df['credit_score'].quantile(0.75)
IQR=Q3-Q1
Lower = Q1 - 1.5*IQR
Upper = Q3 - 1.5*IQR
outliers=df[ (df['credit_score']<Lower) | (df['credit_score']>Upper) ]
outliers

,age,income,credit_score,city,gender,purchases,loan_amount,approved
0,41,16832.970365,646.435852,Hyderabad,Female,2,NaN,0
1,61,24958.920783,772.103370,Bangalore,Female,2,109037.369348,0
2,29,26661.565307,742.165011,Bangalore,Male,3,140981.063625,0
3,40,76918.367953,789.568381,Mumbai,Female,2,265765.286243,0
4,18,33446.160242,650.679984,Delhi,Male,4,187481.809910,0
...,...,...,...,...,...,...,...,...
1044,52,12925.332498,589.778794,Delhi,Male,3,262731.713308,0
1045,57,86596.286095,548.490581,Bangalore,Male,3,124867.554418,0
1047,43,57155.612408,773.613072,Mumbai,Female,1,195876.623311,0
1048,60,35425.789422,549.604263,Mumbai,Male,4,115295.988140,0


In [23]:
median_loan_amount=df['loan_amount'].median()
print(median_loan_amount)

197192.28265723248


In [24]:
Q1=df['loan_amount'].quantile(0.25)
Q3=df['loan_amount'].quantile(0.75)
IQR=Q3-Q1
Lower = Q1 - 1.5*IQR
Upper = Q3 - 1.5*IQR
outliers=df[ (df['loan_amount']<Lower) | (df['loan_amount']>Upper) ]
outliers

,age,income,credit_score,city,gender,purchases,loan_amount,approved
1,61,24958.920783,772.103370,Bangalore,Female,2,109037.369348,0
2,29,26661.565307,742.165011,Bangalore,Male,3,140981.063625,0
3,40,76918.367953,789.568381,Mumbai,Female,2,265765.286243,0
4,18,33446.160242,650.679984,Delhi,Male,4,187481.809910,0
5,57,60504.648191,781.491445,Delhi,Female,3,223926.858722,1
...,...,...,...,...,...,...,...,...
1043,43,48516.655202,702.251411,Mumbai,Male,2,223830.095282,0
1044,52,12925.332498,589.778794,Delhi,Male,3,262731.713308,0
1045,57,86596.286095,548.490581,Bangalore,Male,3,124867.554418,0
1047,43,57155.612408,773.613072,Mumbai,Female,1,195876.623311,0


In [25]:
df['income'].fillna(median_income,inplace=True)
df['credit_score'].fillna(median_credit_score,inplace=True)
df['loan_amount'].fillna(median_loan_amount,inplace=True)
df.isna().sum()

C:\Users\DELL\AppData\Local\Temp\ipykernel_16960\846505939.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['income'].fillna(median_income,inplace=True)
C:\Users\DELL\AppData\Local\Temp\ipykernel_16960\846505939.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example,

age             0
income          0
credit_score    0
city            0
gender          0
purchases       0
loan_amount     0
approved        0
dtype: int64